# nanochat d12 pretraining on Kaggle

This notebook runs the **d12 pretraining** part of the `nanochat` pipeline on Kaggle.

## Before you run it

Make sure the Kaggle notebook is configured with:

- **GPU enabled** (`2x Tesla T4` expected)
- **Internet enabled**
- Your uploaded Kaggle dataset **`nanochat-codes`** attached
- For resume runs, your uploaded Kaggle dataset **`nanochat-d12-pretrain-cache-first-half`** attached

The notebook auto-detects the attached code dataset from this mounted path pattern:
`/kaggle/input/datasets/<your-kaggle-username>/nanochat-codes`.

That mounted dataset is expected to contain the same project files that live under
`kaggle_dataset/nanochat/` in this repository.

## What this notebook does

The notebook runs these stages:

1. Copy the attached `nanochat-codes` dataset into `/kaggle/working/nanochat`
2. Configure caches under `/kaggle/working/nanochat_cache` and `/kaggle/working/huggingface_cache`
3. Install Python dependencies needed by the Kaggle runtime
4. Download a moderate ClimbMix pretraining slice
5. Train and evaluate the tokenizer
6. Pretrain a `d12` base model and save `base_checkpoints/d12_kaggle`
7. Evaluate the `d12` base model on reduced Kaggle-friendly settings
8. Optionally serve the base checkpoint or publish the cache as a Kaggle Dataset


# Instructions for the Full Pretraining Process

Due to the limitations of T4 GPUs, the full training process is split into two half-runs.

## First-half run

In the cell named **“D12 Run Settings”**, set:

```python
PRETRAIN_RUN_MODE = "first_half"
```

## Second-half run

In the first code cell, set:

```python
CACHE_DATASET_NAME = "nanochat-d12-pretrain-cache-first-half"
```

Then, in the cell named **“D12 Run Settings”**, set:

```python
PRETRAIN_RUN_MODE = "second_half"
```

## Smoke run

Before running the full training process, it is recommended to perform a smoke run first.

In the cell named **“D12 Run Settings”**, set:

```python
PRETRAIN_RUN_MODE = "smoke"
```



In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

DATASETS_ROOT = Path('/kaggle/input/datasets')
dataset_candidates = sorted(DATASETS_ROOT.glob('*/nanochat-codes'))
CACHE_DATASET_NAME = 'nanochat-d12-pretrain-cache-first-half' #This is for the second-half run
cache_candidates = sorted(DATASETS_ROOT.glob(f'*/{CACHE_DATASET_NAME}'))

# On Kaggle, this notebook expects an attached dataset named 'nanochat-codes'.
# In this repo, the contents of that mounted dataset folder correspond to the
# local folder 'kaggle_dataset/nanochat/'. The outer 'kaggle_dataset/'
# directory is only the packaging wrapper used with the Kaggle CLI.
assert dataset_candidates, (
    "Could not find an attached Kaggle dataset matching "
    "'/kaggle/input/datasets/<username>/nanochat-codes'"
)
assert len(dataset_candidates) == 1, (
    f"Expected exactly one attached 'nanochat-codes' dataset, found: {dataset_candidates}"
)
assert len(cache_candidates) <= 1, (
    f"Expected at most one attached '{CACHE_DATASET_NAME}' dataset, found: {cache_candidates}"
)
CODE_INPUT = dataset_candidates[0]
CACHE_INPUT = cache_candidates[0] if cache_candidates else None

assert CODE_INPUT.exists(), f'Code dataset not found: {CODE_INPUT}'
print('Code input:', CODE_INPUT)
print('Resume cache input:', CACHE_INPUT if CACHE_INPUT is not None else 'not attached')
print('Top-level code files:', sorted(p.name for p in CODE_INPUT.iterdir()))


Code input: /kaggle/input/datasets/yshuaiqin/nanochat-codes
Resume cache input: not attached
Top-level code files: ['nanochat', 'pyproject.toml', 'scripts', 'tasks', 'tests']


In [2]:
WORK_REPO = Path('/kaggle/working/nanochat')
WORK_CACHE = Path('/kaggle/working/nanochat_cache')
HF_CACHE = Path('/kaggle/working/huggingface_cache')

for path in [WORK_REPO, WORK_CACHE, HF_CACHE]:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

shutil.copytree(CODE_INPUT, WORK_REPO, dirs_exist_ok=True)
if CACHE_INPUT is not None:
    shutil.copytree(CACHE_INPUT, WORK_CACHE, dirs_exist_ok=True)
    print('Copied resume cache into:', WORK_CACHE)

os.environ['NANOCHAT_BASE_DIR'] = str(WORK_CACHE)
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['HF_DATASETS_CACHE'] = str(HF_CACHE / 'datasets')
os.environ['PYTHONFAULTHANDLER'] = '1'

print('Working repo:', WORK_REPO)
print('Nanochat base dir:', WORK_CACHE)
print('HF cache:', HF_CACHE)
print('Repo contents:', sorted(p.name for p in WORK_REPO.iterdir()))


Working repo: /kaggle/working/nanochat
Nanochat base dir: /kaggle/working/nanochat_cache
HF cache: /kaggle/working/huggingface_cache
Repo contents: ['nanochat', 'pyproject.toml', 'scripts', 'tasks', 'tests']


## D12 Run Settings

These defaults are for a first d12 run on Kaggle 2xT4. The model is much larger than d8, so the per-GPU microbatch starts at 4. With 2 GPUs and 1024-token sequences, that gives 8,192 tokens per microbatch and 64 gradient accumulation steps for the 524,288-token total batch. The run downloads 64 train shards so d12 has comfortable margin over its 1.32B-token training target without jumping to the much larger 170-shard GPT-2-style data slice. `PRETRAIN_RUN_MODE` controls smoke, first-half, second-half resume, or full training.


In [3]:
MODEL_DEPTH = 12
MODEL_TAG = 'd12_kaggle'
NUM_GPUS = 2
TRAIN_SHARDS = 64

MAX_SEQ_LEN = 1024
DEVICE_BATCH_SIZE = 4
TOTAL_BATCH_SIZE = 524288
TARGET_PARAM_DATA_RATIO = 12
PRETRAIN_RUN_MODE = 'first_half'  # 'smoke', 'first_half', 'second_half', or 'full'
SMOKE_NUM_ITERATIONS = 50
EXPECTED_D12_SCALING_PARAMS = 110_100_912
TARGET_NUM_ITERATIONS = (TARGET_PARAM_DATA_RATIO * EXPECTED_D12_SCALING_PARAMS) // TOTAL_BATCH_SIZE
SPLIT_STEP = TARGET_NUM_ITERATIONS // 2

EVAL_DEVICE_BATCH_SIZE = 4
EVAL_SPLIT_TOKENS = 262144

world_tokens_per_microbatch = DEVICE_BATCH_SIZE * MAX_SEQ_LEN * NUM_GPUS
assert TOTAL_BATCH_SIZE % world_tokens_per_microbatch == 0, (
    'TOTAL_BATCH_SIZE must be divisible by '
    'DEVICE_BATCH_SIZE * MAX_SEQ_LEN * NUM_GPUS'
)

print('Model tag:', MODEL_TAG)
print('Depth:', MODEL_DEPTH)
print('Train shards:', TRAIN_SHARDS)
print('Pretrain run mode:', PRETRAIN_RUN_MODE)
print('Tokens per microbatch:', world_tokens_per_microbatch)
print('Gradient accumulation steps:', TOTAL_BATCH_SIZE // world_tokens_per_microbatch)
print('Total batch tokens:', TOTAL_BATCH_SIZE)
print('Target iterations:', TARGET_NUM_ITERATIONS)
print('Split checkpoint step:', SPLIT_STEP)


Model tag: d12_kaggle
Depth: 12
Train shards: 64
Pretrain run mode: first_half
Tokens per microbatch: 8192
Gradient accumulation steps: 64
Total batch tokens: 524288
Target iterations: 2520
Split checkpoint step: 1260


## Optional Hugging Face Token

The default pretraining data is public, so this cell normally does not need a token.
Leave `USE_HF_TOKEN = False` for a normal run. Set it to `True` only if you need
authenticated Hugging Face access in your Kaggle session.


In [4]:
USE_HF_TOKEN = False

if USE_HF_TOKEN and not os.environ.get('HF_TOKEN'):
    import getpass
    os.environ['HF_TOKEN'] = getpass.getpass('HF_TOKEN: ')

if os.environ.get('HF_TOKEN'):
    os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
    print('HF_TOKEN is set.')
else:
    print('Skipping HF token. Set USE_HF_TOKEN = True if you need one.')


Skipping HF token. Set USE_HF_TOKEN = True if you need one.


## Install Dependencies

Install the Python packages that are needed in a fresh Kaggle runtime. Kaggle may
already provide some of them, but keeping this cell here makes the notebook easier
to rerun on a new image.


In [5]:
packages = [
    'datasets>=4.0.0',
    'filelock>=3.0.0',
    'psutil>=7.1.0',
    'pyarrow>=19.0.0',
    'python-dotenv>=1.2.1',
    'pyyaml>=6.0.0',
    'regex>=2025.9.1',
    'requests>=2.32.0',
    'rustbpe>=0.1.0',
    'scipy>=1.15.3',
    'tabulate>=0.9.0',
    'tiktoken>=0.11.0',
    'tokenizers>=0.22.0',
    'transformers>=4.57.3',
    'wandb>=0.21.3',
    'zstandard>=0.25.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 36.5 MB/s eta 0:00:00


0

In [6]:
import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device count:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}:', torch.cuda.get_device_name(i))


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.10.0+cu128
CUDA available: True
CUDA device count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## 1. Download a Small Pretraining Slice

This downloads `TRAIN_SHARDS` train shards plus the fixed validation shard into
`/kaggle/working/nanochat_cache/base_data_climbmix`. The d12 default is 64
train shards, up from the old d8 smoke-run value of 20.


In [7]:
cmd = [
    sys.executable,
    '-m', 'nanochat.dataset',
    '-n', str(TRAIN_SHARDS),
    '-w', '4',
]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, check=True)


Running: /usr/bin/python3 -m nanochat.dataset -n 64 -w 4
Target directory: /kaggle/working/nanochat_cache/base_data_climbmix

Successfully downloaded shard_00015.parquet
Successfully downloaded shard_00016.parquet
Successfully downloaded shard_00017.parquet
Successfully downloaded shard_00018.parquet
Successfully downloaded shard_00019.parquet
Successfully downloaded shard_00035.parquet
Successfully downloaded shard_00036.parquet
Successfully downloaded shard_00037.parquet
Successfully downloaded shard_00038.parquet
Successfully downloaded shard_00039.parquet
Successfully downloaded shard_00055.parquet
Successfully downloaded shard_00056.parquet
Successfully downloaded shard_00057.parquet
Successfully downloaded shard_00058.parquet
Successfully downloaded shard_00059.parquet
Successfully downloaded shard_00010.parquet
Successfully downloaded shard_00011.parquet
Successfully downloaded shard_00012.parquet
Successfully downloaded shard_00013.parquet
Successfully downloaded shard_00014.pa

CompletedProcess(args=['/usr/bin/python3', '-m', 'nanochat.dataset', '-n', '64', '-w', '4'], returncode=0)

## 2. Train the Tokenizer

Train the BPE tokenizer from the downloaded pretraining shards. The tokenizer is
saved under `/kaggle/working/nanochat_cache/tokenizer`.


In [8]:
tokenizer_dir = WORK_CACHE / 'tokenizer'
tokenizer_ready = (tokenizer_dir / 'token_bytes.pt').exists()

if tokenizer_ready and CACHE_INPUT is not None:
    print('Tokenizer already present in attached cache; skipping tokenizer training.')
else:
    cmd = [
        sys.executable,
        '-m', 'scripts.tok_train',
    ]

    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=WORK_REPO, check=True)


Running: /usr/bin/python3 -m scripts.tok_train


2026-06-13 06:58:04,094 - rustbpe - INFO - Processing sequences from iterator (buffer_size: 8192)
2026-06-13 07:00:01,716 - rustbpe - INFO - Processed 758523 sequences total, 2098273 unique
2026-06-13 07:00:01,926 - rustbpe - INFO - Starting BPE training: 32503 merges to compute
2026-06-13 07:00:01,926 - rustbpe - INFO - Computing initial pair counts from 2098273 unique sequences
2026-06-13 07:00:04,804 - rustbpe - INFO - Building heap with 19260 unique pairs
2026-06-13 07:00:04,806 - rustbpe - INFO - Starting merge loop
2026-06-13 07:00:08,700 - rustbpe - INFO - Progress: 1% (326/32503 merges) - Last merge: (277, 115) -> 581 (frequency: 613290)
2026-06-13 07:00:09,345 - rustbpe - INFO - Progress: 2% (651/32503 merges) - Last merge: (316, 291) -> 906 (frequency: 250642)
2026-06-13 07:00:09,750 - rustbpe - INFO - Progress: 3% (976/32503 merges) - Last merge: (119, 624) -> 1231 (frequency: 156115)
2026-06-13 07:00:10,047 - rustbpe - INFO - Progress: 4% (1301/32503 merges) - Last merge: (

max_chars: 2,000,000,000
doc_cap: 10,000
vocab_size: 32,768
Training time: 132.04s
Saved tokenizer encoding to /kaggle/working/nanochat_cache/tokenizer/tokenizer.pkl
Saved token_bytes to /kaggle/working/nanochat_cache/tokenizer/token_bytes.pt


## 3. Evaluate the Tokenizer

Run a small tokenizer compression check and compare the trained tokenizer with
GPT-2 and GPT-4 style tokenizers on a few text snippets.


In [9]:
cmd = [
    sys.executable,
    '-m', 'scripts.tok_eval',
]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, check=True)


Running: /usr/bin/python3 -m scripts.tok_eval

Vocab sizes:
GPT-2: 50257
GPT-4: 100277
Ours: 32768

Comparison with GPT-2:
Text Type  Bytes    GPT-2           Ours            Relative     Better    
                    Tokens  Ratio   Tokens  Ratio   Diff %      
-----------------------------------------------------------------------------------------------
news       1819     404     4.50    405     4.49       -0.2%     GPT-2     
korean     893      745     1.20    749     1.19       -0.5%     GPT-2     
code       1259     576     2.19    397     3.17      +31.1%     Ours      
math       1834     936     1.96    911     2.01       +2.7%     Ours      
science    1112     260     4.28    247     4.50       +5.0%     Ours      
fwe-train  2948778  631304  4.67    622480  4.74       +1.4%     Ours      
fwe-val    3024593  653067  4.63    644914  4.69       +1.2%     Ours      

Comparison with GPT-4:
Text Type  Bytes    GPT-4           Ours            Relative     Better    
        

CompletedProcess(args=['/usr/bin/python3', '-m', 'scripts.tok_eval'], returncode=0)

## 4. Pretrain a d12 Base Model

This starts a `d12` base-model pretraining run and writes the checkpoint to
`/kaggle/working/nanochat_cache/base_checkpoints/d12_kaggle`.

The command keeps the Kaggle 2xT4 stability settings from the successful d8 run:

- `NANOCHAT_DTYPE=float16` for Tesla T4
- `NANOCHAT_DISABLE_COMPILE=1` for Kaggle notebook stability
- optimizer communication settings for the Kaggle-safe distributed path
- NCCL settings that avoid unsupported Kaggle interconnect paths
- periodic train-time eval and sampling disabled so pretraining focuses on checkpoint creation
- training progress printed every 50 steps to keep Kaggle output readable


In [10]:
env = os.environ.copy()
env['NANOCHAT_DTYPE'] = 'float16'
env['NANOCHAT_DISABLE_COMPILE'] = '1'
env['NANOCHAT_ADAMW_ALLREDUCE'] = '1'
env['NANOCHAT_SERIAL_OPTIM_COMM'] = '1'
env['OMP_NUM_THREADS'] = '1'
env['NCCL_P2P_DISABLE'] = '1'
env['NCCL_IB_DISABLE'] = '1'
env['NANOCHAT_TRAIN_PRINT_EVERY'] = '50'

cmd = [
    'torchrun',
    '--standalone',
    f'--nproc_per_node={NUM_GPUS}',
    '-m', 'scripts.base_train',
    '--',
    f'--depth={MODEL_DEPTH}',
    f'--model-tag={MODEL_TAG}',
    '--run=dummy',
    f'--max-seq-len={MAX_SEQ_LEN}',
    '--window-pattern=L',
    f'--device-batch-size={DEVICE_BATCH_SIZE}',
    f'--total-batch-size={TOTAL_BATCH_SIZE}',
    f'--target-param-data-ratio={TARGET_PARAM_DATA_RATIO}',
    '--eval-every=-1',
    '--core-metric-every=-1',
    '--sample-every=-1',
    '--save-every=-1',
]

if PRETRAIN_RUN_MODE == 'smoke':
    cmd.append(f'--num-iterations={SMOKE_NUM_ITERATIONS}')
elif PRETRAIN_RUN_MODE == 'first_half':
    cmd.append(f'--num-iterations={SPLIT_STEP}')
elif PRETRAIN_RUN_MODE == 'second_half':
    cmd.append(f'--num-iterations={TARGET_NUM_ITERATIONS}')
    cmd.append(f'--resume-from-step={SPLIT_STEP}')
elif PRETRAIN_RUN_MODE == 'full':
    pass
else:
    raise ValueError(f'Unknown PRETRAIN_RUN_MODE: {PRETRAIN_RUN_MODE}')

if PRETRAIN_RUN_MODE == 'second_half':
    checkpoint_dir = WORK_CACHE / 'base_checkpoints' / MODEL_TAG
    required = [
        checkpoint_dir / f'model_{SPLIT_STEP:06d}.pt',
        checkpoint_dir / f'meta_{SPLIT_STEP:06d}.json',
        checkpoint_dir / f'optim_{SPLIT_STEP:06d}_rank0.pt',
        checkpoint_dir / f'optim_{SPLIT_STEP:06d}_rank1.pt',
    ]
    missing = [path for path in required if not path.exists()]
    assert not missing, 'Missing resume checkpoint files: ' + ', '.join(str(p) for p in missing)

print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)


Running: torchrun --standalone --nproc_per_node=2 -m scripts.base_train -- --depth=12 --model-tag=d12_kaggle --run=dummy --max-seq-len=1024 --window-pattern=L --device-batch-size=4 --total-batch-size=524288 --target-param-data-ratio=12 --eval-every=-1 --core-metric-every=-1 --sample-every=-1 --save-every=-1 --num-iterations=1260


[W613 07:00:25.408034326 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W613 07:00:36.686370683 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W613 07:00:36.687507690 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
2026-06-13 07:00:36,610 - nanochat.common - WARNING - Peak flops undefined for: Tesla T4, MFU will show as 0%
2026-06-13 07:00:36,610 - nanochat.common - INFO - Distributed world size: 2
2026-06-13 07:00:36,611 - nanochat.common - WARNING - Peak flops undefined for: Tesla T4, MFU will show as 0%
[rank0]:W0613 07:01:07.167000 88 torch/_inductor/utils.py:1679] [1/0] Not enough SMs to use max_autotune_gemm mode
[rank1]:W0613 07:01:07.171000 89 torch/_inductor/utils.py:1679] [1/0] Not enough SMs to use max_autotune_gemm mode
2026-06-13 13:24:22,817 - nanochat.checkpoint_manager - INFO - Saved model parameters to: /kaggle/working/nanochat_cache/base_checkpoints/d12_k

CompletedProcess(args=['torchrun', '--standalone', '--nproc_per_node=2', '-m', 'scripts.base_train', '--', '--depth=12', '--model-tag=d12_kaggle', '--run=dummy', '--max-seq-len=1024', '--window-pattern=L', '--device-batch-size=4', '--total-batch-size=524288', '--target-param-data-ratio=12', '--eval-every=-1', '--core-metric-every=-1', '--sample-every=-1', '--save-every=-1', '--num-iterations=1260'], returncode=0)

## 5. Evaluate the d12 Base Model

This loads `base_checkpoints/d12_kaggle` and runs a reduced 2-GPU evaluation.
The settings are intentionally small enough for Kaggle:

- run `core`, `bpb`, and `sample`
- limit CORE examples with `--max-per-task=50`
- reduce BPB tokens with `--split-tokens=262144`

These numbers are useful for a smoke test and rough sanity check. They are not
intended as final benchmark results.


In [11]:
env = os.environ.copy()
env['NANOCHAT_DTYPE'] = 'float16'
env['NANOCHAT_DISABLE_COMPILE'] = '1'
env['OMP_NUM_THREADS'] = '1'
env['NCCL_P2P_DISABLE'] = '1'
env['NCCL_IB_DISABLE'] = '1'

cmd = [
    'torchrun',
    '--standalone',
    f'--nproc_per_node={NUM_GPUS}',
    '-m', 'scripts.base_eval',
    '--',
    '--eval', 'core,bpb,sample',
    '--model-tag', MODEL_TAG,
    '--max-per-task', '50',
    '--device-batch-size', str(EVAL_DEVICE_BATCH_SIZE),
    '--split-tokens', str(EVAL_SPLIT_TOKENS),
]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)


Running: torchrun --standalone --nproc_per_node=2 -m scripts.base_eval -- --eval core,bpb,sample --model-tag d12_kaggle --max-per-task 50 --device-batch-size 4 --split-tokens 262144


[W613 13:24:31.273385502 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W613 13:24:36.731960585 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W613 13:24:36.819361105 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
2026-06-13 13:24:36,593 - nanochat.common - INFO - Distributed world size: 2
2026-06-13 13:24:36,594 - nanochat.checkpoint_manager - INFO - Loading model from /kaggle/working/nanochat_cache/base_checkpoints/d12_kaggle with step 1260
2026-06-13 13:24:37,609 - nanochat.checkpoint_manager - INFO - Building model with config: {'sequence_len': 1024, 'vocab_size': 32768, 'n_layer': 12, 'n_head': 6, 'n_kv_head': 6, 'n_embd': 768, 'window_pattern': 'L'}


CompletedProcess(args=['torchrun', '--standalone', '--nproc_per_node=2', '-m', 'scripts.base_eval', '--', '--eval', 'core,bpb,sample', '--model-tag', 'd12_kaggle', '--max-per-task', '50', '--device-batch-size', '4', '--split-tokens', '262144'], returncode=0)

## 6. Inspect Pretraining Artifacts

Print the cache size and a shallow file listing so you can confirm that data,
tokenizer files, reports, and base checkpoints were written where expected.


In [12]:
print('Cache size:')
subprocess.run(['du', '-sh', str(WORK_CACHE)], check=False)

print()
print('Top-level cache contents:')
subprocess.run(['find', str(WORK_CACHE), '-maxdepth', '2', '-type', 'f'], check=False)


Cache size:

Top-level cache contents:


CompletedProcess(args=['find', '/kaggle/working/nanochat_cache', '-maxdepth', '2', '-type', 'f'], returncode=0)

## 7. Optional: Serve the d12 Base Model

Run this only after `base_checkpoints/d12_kaggle` exists. The cell is disabled by
default; set `RUN_SERVER = True` when you want to launch the web server.

This serves the pretrained base model, not an SFT chat model. Treat it as a
loading and generation smoke test rather than a polished assistant demo.


In [13]:
# Optional: start a NanoChat web server for the pretrained base checkpoint.
# Set RUN_SERVER = True and run this cell when you want to serve the model.
RUN_SERVER = False

if not RUN_SERVER:
    print('Skipping server start. Set RUN_SERVER = True to launch it.')
else:
    subprocess.check_call([
        sys.executable,
        '-m', 'pip', 'install', '-q',
        'fastapi>=0.115.0',
        'pydantic>=2.0.0',
        'uvicorn[standard]>=0.30.0',
    ])

    try:
        server.terminate()
        server.wait(timeout=10)
        print('Stopped old server.')
    except NameError:
        print('No existing server variable found.')
    except Exception as e:
        print('Could not stop old server cleanly:', e)
        try:
            server.kill()
            print('Killed old server.')
        except Exception:
            pass

    env = os.environ.copy()
    env['NANOCHAT_DTYPE'] = 'float16'
    env['OMP_NUM_THREADS'] = '1'
    env['NCCL_P2P_DISABLE'] = '1'
    env['NCCL_IB_DISABLE'] = '1'

    cmd = [
        sys.executable,
        '-m', 'scripts.chat_web',
        '--source', 'base',
        '--model-tag', MODEL_TAG,
        '--num-gpus', '1',
        '--host', '0.0.0.0',
        '--port', '8000',
    ]

    print('Running:', ' '.join(cmd))
    server = subprocess.Popen(cmd, cwd=WORK_REPO, env=env)
    print('Server starting on http://127.0.0.1:8000')


Skipping server start. Set RUN_SERVER = True to launch it.


## 8. Optional: Test the Server

Run this after the server has started. The cell is disabled by default; set
`TEST_SERVER = True` to send one streaming request to `http://127.0.0.1:8000`.

Because the checkpoint is only pretrained, output quality is not expected to be
chat-like. This cell only checks that the server can accept a request and stream tokens.


In [14]:
# Optional: smoke-test the local web server.
# Set TEST_SERVER = True after the server has started.
TEST_SERVER = False

if not TEST_SERVER:
    print('Skipping server test. Set TEST_SERVER = True after starting the server.')
else:
    import json
    import time
    import requests

    BASE_URL = 'http://127.0.0.1:8000'

    for attempt in range(30):
        try:
            health = requests.get(f'{BASE_URL}/health', timeout=5)
            health.raise_for_status()
            print('Health:', health.json())
            break
        except Exception as e:
            if attempt == 29:
                raise RuntimeError('Server did not become healthy.') from e
            time.sleep(2)

    payload = {
        'messages': [
            {'role': 'user', 'content': 'Write one short sentence about machine learning.'}
        ],
        'temperature': 0.8,
        'top_k': 50,
        'max_tokens': 80,
    }

    print('Assistant:', end=' ', flush=True)
    with requests.post(f'{BASE_URL}/chat/completions', json=payload, stream=True, timeout=120) as response:
        response.raise_for_status()
        for line in response.iter_lines(decode_unicode=True):
            if not line or not line.startswith('data: '):
                continue
            data = json.loads(line[len('data: '):])
            if data.get('done'):
                break
            print(data.get('token', ''), end='', flush=True)
    print()


Skipping server test. Set TEST_SERVER = True after starting the server.


## 9. Optional: Stop the Server

Use this when you are done testing the local server. Set `STOP_SERVER = True`
and run the cell to terminate the background process.


In [15]:
# Optional: stop the server started above.
STOP_SERVER = False

if not STOP_SERVER:
    print('Skipping server stop. Set STOP_SERVER = True to stop it.')
else:
    try:
        server.terminate()
        server.wait(timeout=10)
        print('Server stopped.')
    except NameError:
        print('No server variable found.')
    except Exception as e:
        print('Could not stop server cleanly:', e)
        try:
            server.kill()
            print('Server killed.')
        except Exception:
            pass


Skipping server stop. Set STOP_SERVER = True to stop it.


## 10. Optional: Save the Pretraining Cache as a Kaggle Dataset

Use this only if you want to publish `/kaggle/working/nanochat_cache` as a
reusable Kaggle Dataset for later SFT/evaluation notebooks.

The cell is safe to leave in the notebook: it skips upload until `DATASET_ID` is
filled with a value like `your-kaggle-username/nanochat-d12-pretrain-cache`.


In [16]:
# Optional: save /kaggle/working/nanochat_cache as a Kaggle Dataset.
# For the two-run workflow, this cell prunes downloaded data and keeps only
# the tokenizer plus the checkpoint needed to resume.
# Fill this in before running the cell, for example:
# DATASET_ID = 'your-kaggle-username/nanochat-d12-pretrain-cache'
import json

CACHE_DIR = Path('/kaggle/working/nanochat_cache')
if PRETRAIN_RUN_MODE == 'first_half':
         #'your-kaggle-username/nanochat-d12-pretrain-cache-first-half' when you do the first-half-run
         DATASET_ID = 'yshuaiqin/nanochat-d12-pretrain-cache-first-half'
         TITLE = 'nanochat d12 pretrain cache first half'  
elif PRETRAIN_RUN_MODE == 'second_half':
         #'your-kaggle-username/nanochat-d12-pretrain-cache' when you do the last-half-run
         DATASET_ID = 'yshuaiqin/nanochat-d12-pretrain-cache'
         TITLE = 'nanochat d12 pretrain cache'
else:
            DATASET_ID = ''  #'' when you do the smoke or full run, since those are not meant for resuming
            TITLE = 'nanochat d12 pretrain cache'
VERSION_MESSAGE = f'Save full nanochat_cache after d12 pretraining with {TRAIN_SHARDS} train shards'
PRUNE_FOR_RESUME_UPLOAD = True
if PRETRAIN_RUN_MODE == 'first_half':
    KEEP_CHECKPOINT_STEP = SPLIT_STEP
elif PRETRAIN_RUN_MODE in {'second_half', 'full'}:
    KEEP_CHECKPOINT_STEP = TARGET_NUM_ITERATIONS
else:
    KEEP_CHECKPOINT_STEP = None

if not DATASET_ID:
    print('Skipping upload. Set DATASET_ID in this cell and rerun it if you want to publish the cache.')
else:
    assert '/' in DATASET_ID, "DATASET_ID should look like '<username>/<dataset-slug>'."
    assert CACHE_DIR.exists(), f'{CACHE_DIR} does not exist'
    assert any(CACHE_DIR.iterdir()), f'{CACHE_DIR} is empty'

    if PRUNE_FOR_RESUME_UPLOAD:
        if KEEP_CHECKPOINT_STEP is None:
            raise RuntimeError('Do not publish a resume cache from smoke mode. Run first_half first.')

        data_dir = CACHE_DIR / 'base_data_climbmix'
        if data_dir.exists():
            shutil.rmtree(data_dir)
            print('Pruned downloaded pretraining data:', data_dir)

        checkpoint_dir = CACHE_DIR / 'base_checkpoints' / MODEL_TAG
        assert checkpoint_dir.exists(), f'Missing checkpoint dir: {checkpoint_dir}'

        def checkpoint_step(path):
            for part in path.stem.split('_'):
                if part.isdigit():
                    return int(part)
            return None

        for pattern in ['model_*.pt', 'meta_*.json', 'optim_*.pt']:
            for path in checkpoint_dir.glob(pattern):
                if checkpoint_step(path) != KEEP_CHECKPOINT_STEP:
                    path.unlink()
                    print('Pruned old checkpoint file:', path.name)

        required = [
            checkpoint_dir / f'model_{KEEP_CHECKPOINT_STEP:06d}.pt',
            checkpoint_dir / f'meta_{KEEP_CHECKPOINT_STEP:06d}.json',
            checkpoint_dir / f'optim_{KEEP_CHECKPOINT_STEP:06d}_rank0.pt',
            checkpoint_dir / f'optim_{KEEP_CHECKPOINT_STEP:06d}_rank1.pt',
        ]
        missing = [path for path in required if not path.exists()]
        assert not missing, 'Missing resume cache files: ' + ', '.join(str(p) for p in missing)
        print('Resume cache checkpoint step:', KEEP_CHECKPOINT_STEP)

    metadata = {
        'title': TITLE,
        'id': DATASET_ID,
        'licenses': [
            {'name': 'CC0-1.0'}
        ]
    }

    metadata_path = CACHE_DIR / 'dataset-metadata.json'
    metadata_path.write_text(json.dumps(metadata, indent=2))

    print('Dataset metadata:')
    print(metadata_path.read_text())

    print()
    print('Folder size:')
    subprocess.run(['du', '-sh', str(CACHE_DIR)], check=False)

    status = subprocess.run(
        ['kaggle', 'datasets', 'status', DATASET_ID],
        text=True,
        capture_output=True,
    )

    if status.returncode == 0:
        print()
        print(f'Dataset already exists: {DATASET_ID}')
        print('Creating a new dataset version...')
        cmd = [
            'kaggle', 'datasets', 'version',
            '-p', str(CACHE_DIR),
            '-m', VERSION_MESSAGE,
            '--dir-mode', 'tar',
        ]
    else:
        print()
        print(f'Dataset does not exist yet: {DATASET_ID}')
        print('Creating a new private dataset...')
        cmd = [
            'kaggle', 'datasets', 'create',
            '-p', str(CACHE_DIR),
            '--dir-mode', 'tar',
        ]

    print()
    print('Running:', ' '.join(cmd))
    result = subprocess.run(cmd, text=True)

    if result.returncode != 0:
        raise RuntimeError('Kaggle Dataset upload failed.')

    print()
    print('Done.')
    print(f'Dataset URL: https://www.kaggle.com/datasets/{DATASET_ID}')


Pruned downloaded pretraining data: /kaggle/working/nanochat_cache/base_data_climbmix
Resume cache checkpoint step: 1260
Dataset metadata:
{
  "title": "nanochat d12 pretrain cache first half",
  "id": "yshuaiqin/nanochat-d12-pretrain-cache-first-half",
  "licenses": [
    {
      "name": "CC0-1.0"
    }
  ]
}

Folder size:

Dataset does not exist yet: yshuaiqin/nanochat-d12-pretrain-cache-first-half
Creating a new private dataset...

Running: kaggle datasets create -p /kaggle/working/nanochat_cache --dir-mode tar


100%|██████████| 540k/540k [00:00<00:00, 807kB/s]
100%|██████████| 24.9M/24.9M [00:01<00:00, 24.3MB/s]
100%|██████████| 20.0k/20.0k [00:00<00:00, 29.9kB/s]
100%|██████████| 161M/161M [00:02<00:00, 59.4MB/s]
100%|██████████| 10.0k/10.0k [00:00<00:00, 14.5kB/s]
100%|██████████| 4.38G/4.38G [01:10<00:00, 67.0MB/s]



Done.
Dataset URL: https://www.kaggle.com/datasets/yshuaiqin/nanochat-d12-pretrain-cache-first-half
